# Hazard Overlay
This tutorial guides you through performing a hazard overlay with a network using RA2CE.
It assumes you already have a road network. If not, please first complete the [Network tutorial](network).

## Hazard Map
A hazard map is geospatial data representing the extent and severity of a hazard in a specific area. RA2CE supports raster formats such as GeoTIFF.

> **Note:** RA2CE is hazard agnostic: any hazard that can be represented in a raster format (flood, landslide, wildfire) can be used.

## What is hazard overlay?
The road network is represented as links and nodes, and the hazard map as a raster grid. By overlaying the network onto the hazard map, each road segment is assigned hazard values from the intersecting cells. This allows RA2CE to quantify network exposure and supports risk assessment and accessibility studies.

## Step 1. Import the Required Packages

In [6]:
from pathlib import Path
import geopandas as gpd

from ra2ce.network.network_config_data.enums.aggregate_wl_enum import AggregateWlEnum
from ra2ce.network.network_config_data.network_config_data import (
    NetworkSection, NetworkConfigData, HazardSection, CleanupSection
)
from ra2ce.network.network_config_data.enums.source_enum import SourceEnum
from ra2ce.ra2ce_handler import Ra2ceHandler

# Root project directory
root_dir = Path('data', 'hazard_overlay')
network_path = root_dir / "network"
hazard_path = root_dir / "hazard"

## Step 2. Define Network and Hazard Sections
Specify the network and hazard map sections in the configuration. Provide the CRS of the hazard map and select an aggregation method for hazard values.

In [7]:
gdf = gpd.read_file(network_path.joinpath("base_shapefile.shp"))
gdf_reduced = gdf[['fid']]
import numpy as np
gdf_reduced['hazard_value'] = gdf_reduced['fid'] * 100  # Example hazard values

# save to csv
output_csv_path = hazard_path / "hazard_file_table.csv"
gdf_reduced.to_csv(output_csv_path, index=False)

C:\Users\hauth\AppData\Local\Temp\ipykernel_38092\791106954.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gdf_reduced['hazard_value'] = gdf_reduced['fid'] * 100  # Example hazard values


In [8]:
# Define the network section
network_section = NetworkSection(
    source=SourceEnum.SHAPEFILE,
    primary_file=network_path.joinpath("base_shapefile.shp"),
    save_gpkg=True,
    file_id='rfid_d' # This cant be fid because it's reserved in GPKG; 
)

# Define the hazard section
hazard_section = HazardSection(
    hazard_map=[hazard_path / "hazard_file_table.csv"],
    aggregate_wl=AggregateWlEnum.MEAN,
    hazard_field_name="hazard_value",
    hazard_id="fid",  # generated from the network file with the column name "fid"
    hazard_crs="EPSG:32736",
)

# Build the full configuration
network_config_data = NetworkConfigData(
    root_path=root_dir,
    static_path=root_dir.joinpath('static'),
    network=network_section,
    hazard=hazard_section
)

The network is segment by default into 100m segments. This can be modified using the attribute `segmentation_length` from [CleanupSection](../api/ra2ce.network.network_config_data.html#ra2ce.network.network_config_data.network_config_data.CleanupSection){.api-ref}.

In [9]:
# Define the network section
network_section = CleanupSection(
    segmentation_length=1000 # in meters
)

## Step 3. Initialize and Configure RA2CE
Generate both the base network and the overlaid hazard network. Results will be stored in `static/output_graph`.

In [10]:
handler = Ra2ceHandler.from_config(network=network_config_data, analysis=None)
handler.configure()

100%|██████████| 4217/4217 [00:00<00:00, 323636.46it/s]
2026-04-22 11:31:14 AM - [avg_speed_calculator.py:175] - root - WARNING - No valid file found with average speeds data\hazard_overlay\static\output_graph\avg_speed.csv, calculating and saving them instead.
2026-04-22 11:31:14 AM - [avg_speed_calculator.py:175] - root - WARNING - No valid file found with average speeds data\hazard_overlay\static\output_graph\avg_speed.csv, calculating and saving them instead.
2026-04-22 11:31:14 AM - [avg_speed_calculator.py:150] - root - WARNING - Default speed have been assigned to road type [<RoadTypeEnum.SECONDARY: 7>]. Please check the average speed CSV, enter the right average speed for this road type and run RA2CE again.
2026-04-22 11:31:14 AM - [avg_speed_calculator.py:150] - root - WARNING - Default speed have been assigned to road type [<RoadTypeEnum.SECONDARY: 7>]. Please check the average speed CSV, enter the right average speed for this road type and run RA2CE again.
2026-04-22 11:31:1

## Step 4. Inspect Hazard Overlay Results
Once the analysis is complete, hazard overlay results are stored in the `output_graph` folder. Files containing `_hazard` hold the overlay results for each road segment. For example, `EV1_ma` represents the maximum flood depth for Event 1.

In [ ]:
hazard_output = root_dir / "static" / "output_graph" / "base_graph_hazard_edges.gpkg"
hazard_gdf = gpd.read_file(hazard_output, driver = "GPKG")
hazard_gdf.head()